# Where do the four algorithms live, and in which basis?

Three analyses on the 4-task model that grokked div + add + max + parity at P=149, WD=0.3:

1. **Each algorithm in each basis** — same final model, residuals at L2 projected into each task's privileged basis (dlog for div, natural for add, top-SVD for max, Nyquist for parity). On-diagonal: clean structure. Off-diagonal: noise. Cross-basis 4×4 grid.
2. **Where the algorithm lives along the forward pass** — pick 9 points (tok_emb_a, tok_emb_b, L0_eq, attn0_out_eq, ffn0_mid_eq, L1_eq, attn1_out_eq, ffn1_mid_eq, L2_eq) and project the per-class residuals into each task's basis at each location. 4×9 grid.
3. **Parity ↔ add Nyquist coupling** — trace the power at frequency k=74 in W_U across training. Parity locks first, leaving add stuck on the 50% plateau until the coupling breaks.

Loads the seed-1000 final checkpoint and snapshots from `./out/seed_1000/` (produced by `train_5seeds.ipynb`). For a fast path, the cell at the top will fetch them from the Hugging Face Hub instead.

In [ ]:
# Cell 1 — setup + load checkpoint (local ./out preferred, HF Hub fallback for final.pt only)
import json, gc
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from numpy.linalg import svd
import matplotlib.pyplot as plt

try:
    from huggingface_hub import hf_hub_download
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
    from huggingface_hub import hf_hub_download

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

P = 149
OP_DIV, OP_ADD, OP_MAX, OP_PARITY, EQ = P, P+1, P+2, P+3, P+4
VOCAB = P + 5
TASKS = ['div', 'add', 'max', 'parity']
TASK_OP = {'div': OP_DIV, 'add': OP_ADD, 'max': OP_MAX, 'parity': OP_PARITY}
TASK_COLORS = {'div': '#D45A2A', 'add': '#3A6E8C', 'max': '#5E9C5C', 'parity': '#9C5EBE'}

LOCAL_FINAL = Path('./out/seed_1000/final.pt')
LOCAL_SNAP_DIR = Path('./out/seed_1000/snapshots')
OUT_DIR = Path('./out/analysis_seed_1000'); OUT_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_FINAL.exists():
    FINAL_PATH = LOCAL_FINAL
    print(f'Using local checkpoint: {FINAL_PATH}')
else:
    FINAL_PATH = Path(hf_hub_download(
        repo_id='NikolayYudin/manifold-features-data',
        filename='four-algorithms/seed_1000_final.pt',
        repo_type='dataset',
    ))
    print(f'Using HF Hub checkpoint: {FINAL_PATH}')
    print('  (note: snapshots are not on the Hub; the Nyquist-trace analysis below needs them. Run train_5seeds.ipynb to produce snapshots first.)')

BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.5, 'axes.labelcolor': FG,
    'axes.titlecolor': FG, 'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.6,
})

# discrete log table for div basis
def is_gen(g, p):
    seen, x = set(), 1
    for _ in range(p - 1):
        x = (x * g) % p
        if x in seen: return False
        seen.add(x)
    return True
g_gen = next(g for g in range(2, P) if is_gen(g, P))
PM1 = P - 1
exp_table = np.zeros(PM1, dtype=np.int64); x = 1
for t in range(PM1): exp_table[t] = x; x = (x * g_gen) % P
dlog_arr = np.zeros(P, dtype=np.int64)
for t in range(PM1): dlog_arr[exp_table[t]] = t
print(f'g={g_gen}, PM1={PM1}')

In [ ]:
# Cell 2 — architecture (with multi-location capture helper)
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.scale

class GrokBlock(nn.Module):
    def __init__(self, d, nh, dropout=0.2):
        super().__init__()
        self.norm1 = RMSNorm(d); self.attn = nn.MultiheadAttention(d, nh, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d); self.w1 = nn.Linear(d, 4*d, bias=False)
        self.w2 = nn.Linear(4*d, d, bias=False); self.w3 = nn.Linear(d, 4*d, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x); o, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(o)
        h2 = self.norm2(x); gate = F.silu(self.w1(h2))
        return x + self.dropout(self.w2(gate * self.w3(h2)))

class GrokModelShared(nn.Module):
    def __init__(self, vocab=VOCAB, d=384, nh=12, n_layers=2, out_p=P):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d); self.pos_emb = nn.Embedding(4, d)
        self.blocks = nn.ModuleList([GrokBlock(d, nh) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        self.head = nn.Linear(d, out_p, bias=False)
    def to_L2(self, a, op_token, b):
        B = a.size(0)
        op = torch.full((B,), op_token, device=a.device, dtype=torch.long)
        eq = torch.full((B,), EQ, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        return self.norm_f(x)[:, -1, :]
    @torch.no_grad()
    def capture_all_locations(self, a, op_token, b):
        B = a.size(0)
        op = torch.full((B,), op_token, device=a.device, dtype=torch.long)
        eq = torch.full((B,), EQ, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        out = {'tok_emb_a': self.tok_emb(a),
               'tok_emb_b': self.tok_emb(b),
               'L0_eq':     x[:, -1, :].clone()}
        bl0 = self.blocks[0]; h0 = bl0.norm1(x); o0, _ = bl0.attn(h0, h0, h0, need_weights=False)
        x = x + o0
        out['attn0_out_eq'] = o0[:, -1, :].clone()
        h2_0 = bl0.norm2(x); gate0 = F.silu(bl0.w1(h2_0))
        out['ffn0_mid_eq'] = (gate0 * bl0.w3(h2_0))[:, -1, :].clone()
        x = x + bl0.w2(gate0 * bl0.w3(h2_0))
        out['L1_eq'] = x[:, -1, :].clone()
        bl1 = self.blocks[1]; h1 = bl1.norm1(x); o1, _ = bl1.attn(h1, h1, h1, need_weights=False)
        x = x + o1
        out['attn1_out_eq'] = o1[:, -1, :].clone()
        h2_1 = bl1.norm2(x); gate1 = F.silu(bl1.w1(h2_1))
        out['ffn1_mid_eq'] = (gate1 * bl1.w3(h2_1))[:, -1, :].clone()
        x = x + bl1.w2(gate1 * bl1.w3(h2_1))
        out['L2_eq'] = self.norm_f(x)[:, -1, :].clone()
        return {k: v.cpu().numpy() for k, v in out.items()}

ckpt = torch.load(FINAL_PATH, map_location=device, weights_only=False)
m = GrokModelShared(d=384).to(device)
m.load_state_dict(ckpt['state_dict']); m.eval()
print(f'Loaded. Final accs: {ckpt["final_accs"]}')

def make_task_data(task):
    aa, bb, cc = [], [], []
    for a in range(P):
        for b in range(P):
            if task == 'div' and b == 0: continue
            if task == 'div':    c = (a * pow(b, P-2, P)) % P
            elif task == 'add':  c = (a + b) % P
            elif task == 'max':  c = max(a, b)
            elif task == 'parity': c = (a + b) % 2
            aa.append(a); bb.append(b); cc.append(c)
    return (torch.tensor(aa, device=device), torch.tensor(bb, device=device),
            torch.tensor(cc, device=device))

@torch.no_grad()
def per_class_mean(loc_per_pair, c_np, n_cls):
    M_ = np.zeros((n_cls, loc_per_pair.shape[1]), dtype=np.float32)
    for cls in range(n_cls):
        idx = (c_np == cls)
        if idx.any(): M_[cls] = loc_per_pair[idx].mean(0)
    return M_

def project_in_basis(M_class, task):
    """Return 2D projection of per-class means in the task's privileged basis."""
    if task == 'parity':
        # rank-1: project onto first SVD direction, plot as (x, 0)
        Mc = M_class - M_class.mean(0, keepdims=True)
        _, _, Vt = svd(Mc, full_matrices=False)
        return Mc @ Vt[:2].T
    elif task == 'div':
        # reorder rows by dlog, then natural Fourier on the reordered rows
        valid = list(range(1, P))   # class 0 excluded for div (no element)
        Mc = M_class.copy()
        # build a vector index where slot i is the class whose dlog == i
        order = exp_table.tolist()    # length PM1=148; exp_table[i] is the class with dlog i
        rows = Mc[order]              # (PM1, d)
        rows = rows - rows.mean(0, keepdims=True)
        _, _, Vt = svd(rows, full_matrices=False)
        return rows @ Vt[:2].T
    else:
        # natural order Fourier-friendly SVD
        Mc = M_class - M_class.mean(0, keepdims=True)
        _, _, Vt = svd(Mc, full_matrices=False)
        return Mc @ Vt[:2].T

In [ ]:
# Cell 3 — Figure 1: each algorithm in each basis (cross-basis 4×4 grid)
# Collect per-task L2 manifolds
L2_manifolds = {}
for t in TASKS:
    a, b, c = make_task_data(t)
    with torch.no_grad():
        h = m.to_L2(a, TASK_OP[t], b).cpu().numpy()
    c_np = c.cpu().numpy()
    n_cls = 2 if t == 'parity' else P
    L2_manifolds[t] = per_class_mean(h, c_np, n_cls)

fig, axes = plt.subplots(len(TASKS), len(TASKS), figsize=(12, 12), dpi=120)
for i, row_task in enumerate(TASKS):       # the data
    for j, col_task in enumerate(TASKS):    # the basis
        ax = axes[i, j]
        M = L2_manifolds[row_task]
        proj = project_in_basis(M, col_task)
        ax.scatter(proj[:, 0], proj[:, 1],
                   c=np.arange(M.shape[0]), cmap='twilight', s=15, edgecolor='none')
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(f'basis: {col_task}', fontsize=10, color=TASK_COLORS[col_task])
        if j == 0:
            ax.set_ylabel(f'data: {row_task}', fontsize=10, color=TASK_COLORS[row_task])
plt.suptitle('Each algorithm in each basis — diagonal: structure; off-diagonal: static', fontsize=11, y=0.995)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_cross_basis_4x4.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4 — Figure 2: where do the algorithms live (4 tasks × 9 forward-pass locations)
LOCATIONS = ['tok_emb_a', 'tok_emb_b', 'L0_eq', 'attn0_out_eq', 'ffn0_mid_eq',
             'L1_eq', 'attn1_out_eq', 'ffn1_mid_eq', 'L2_eq']

loc_manifolds = {t: {} for t in TASKS}
for t in TASKS:
    a, b, c = make_task_data(t)
    captures = m.capture_all_locations(a, TASK_OP[t], b)
    c_np = c.cpu().numpy(); n_cls = 2 if t == 'parity' else P
    for loc in LOCATIONS:
        loc_manifolds[t][loc] = per_class_mean(captures[loc], c_np, n_cls)

fig, axes = plt.subplots(len(TASKS), len(LOCATIONS), figsize=(22, 10), dpi=120)
for i, t in enumerate(TASKS):
    for j, loc in enumerate(LOCATIONS):
        ax = axes[i, j]
        M = loc_manifolds[t][loc]
        proj = project_in_basis(M, t)
        ax.scatter(proj[:, 0], proj[:, 1],
                   c=np.arange(M.shape[0]), cmap='twilight', s=8, edgecolor='none')
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        if i == 0: ax.set_title(loc, fontsize=8, rotation=20)
        if j == 0: ax.set_ylabel(t, fontsize=10, color=TASK_COLORS[t])
plt.suptitle('Where each algorithm lives along the forward pass (proper basis per row)', fontsize=11, y=1.005)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2_multilocation_4x9.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 5 — Figure 3: parity ↔ add Nyquist coupling trace at k=74 (requires snapshots)
if not LOCAL_SNAP_DIR.exists():
    print(f'No snapshots at {LOCAL_SNAP_DIR}. Run train_5seeds.ipynb with SAVE_SNAPSHOTS=True first.')
    print('Skipping Nyquist trace.')
else:
    snap_files = sorted(LOCAL_SNAP_DIR.glob('step_*.pt'))
    snap_steps = [int(f.stem.split('_')[1]) for f in snap_files]
    print(f'Found {len(snap_files)} snapshots, steps {snap_steps[0]} .. {snap_steps[-1]}')

    @torch.no_grad()
    def add_k74_power(sd):
        m_tmp = GrokModelShared(d=384).to(device)
        m_tmp.load_state_dict({k: v.to(device) for k, v in sd.items()}, strict=False)
        m_tmp.eval()
        head_W = m_tmp.head.weight.detach().cpu().numpy()  # [P, d]
        # power at frequency k=74 in the natural-order Fourier basis along class axis
        Mc = head_W - head_W.mean(0, keepdims=True)
        fft = np.fft.rfft(Mc, axis=0)
        k74_power = float(np.abs(fft[74])**2).sum() if False else float((np.abs(fft[74])**2).sum())
        del m_tmp; gc.collect()
        return k74_power

    metrics_per_step = {r['step']: {f'test_{t}': r[f'test_{t}'] for t in TASKS} for r in ckpt['metrics']}
    parity_accs, add_accs, k74_powers = [], [], []
    for step, f in zip(snap_steps, snap_files):
        # nearest metric record (snapshots may not be at the same steps as METRICS_EVERY samples)
        near = min(metrics_per_step.keys(), key=lambda s: abs(s - step))
        parity_accs.append(metrics_per_step[near]['test_parity'])
        add_accs.append(metrics_per_step[near]['test_add'])
        sd = torch.load(f, map_location='cpu')
        k74_powers.append(add_k74_power(sd))

    fig, axes = plt.subplots(2, 1, figsize=(11, 5), dpi=120, sharex=True)
    axes[0].plot(snap_steps, parity_accs, color=TASK_COLORS['parity'], linewidth=1.6, label='parity test acc')
    axes[0].plot(snap_steps, add_accs, color=TASK_COLORS['add'], linewidth=1.6, label='add test acc')
    axes[0].axhline(0.5, color=MUTED, linestyle=':', linewidth=0.5)
    axes[0].set_ylabel('test accuracy'); axes[0].legend(frameon=False, loc='center right')
    axes[1].plot(snap_steps, k74_powers, color='#3A6E8C', linewidth=1.4)
    axes[1].set_ylabel('power at k=74 in W_U (Nyquist)'); axes[1].set_xlabel('step')
    plt.suptitle('Nyquist coupling: parity captures k=74 first; add unsticks once k=74 reorganizes', fontsize=10)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'fig3_nyquist_coupling.png', dpi=120, bbox_inches='tight')
    plt.show()

    with open(OUT_DIR / 'nyquist_trace.json', 'w') as f:
        json.dump({'snap_steps': snap_steps,
                   'parity_acc': parity_accs,
                   'add_acc': add_accs,
                   'k74_power_W_U': k74_powers}, f, indent=2)

## What these figures show

- **Fig 1** — same residuals at L2, projected into each task's privileged basis. Diagonal: each task has clean geometric structure in its own basis. Off-diagonal: the same data through another task's basis looks like noise. There is no universal coordinate system inside the model.
- **Fig 2** — each row is a task, each column is a forward-pass location. Add's ring is present already in `tok_emb_b`. Div's ring crystallises at `attn0_out_eq` and persists. Max's value-monotonic structure is everywhere. Parity's two points pull apart progressively. Each algorithm has a different location signature.
- **Fig 3** — power at frequency k=74 (Nyquist for P=149) in W_U over training, overlaid with parity and add test accuracy. Parity locks the k=74 direction first, leaving add stuck at the 50% plateau. Once the k=74 representation reorganises (the model finds a second direction for add), add unsticks.

Outputs in `./out/analysis_seed_1000/`.